# Notebook 12 — Clasificación con Regresión Logística 🎯

En el **NB08** entrenaste tu primera regresión lineal — predijiste un **número continuo** (`fare`). Hoy das el salto al otro gran tipo de problema supervisado: **clasificación**.

La pregunta cambia de **"¿cuánto?"** a **"¿cuál?"**:

```
    Regresión (NB08)              Clasificación (hoy)
    ────────────────              ───────────────────
    ¿Cuánto pagó este             ¿Sobrevivió este
    pasajero por el billete?      pasajero?  (sí / no)
```

El target ya no es un float, sino una categoría: **`survived = 0` (murió) ó `1` (sobrevivió)**.

## Objetivos de aprendizaje

1. Entender la diferencia entre **regresión** y **clasificación**.
2. Intuición de la **función sigmoide** y por qué da una **probabilidad** ∈ [0, 1].
3. Entrenar un modelo `LogisticRegression` con scikit-learn.
4. Usar **`.predict()`** (clase predicha) vs **`.predict_proba()`** (probabilidad).
5. Aplicar todo lo aprendido en los NB09–NB11: encoding + feature engineering + escalado.

---

## 1. Setup — pipeline completo de los NB09–NB11

Aplicamos todo lo aprendido: limpieza, encoding y escalado.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Same cleanup as previous notebooks
df = sns.load_dataset("titanic")
df_clean = df.drop(columns=["deck"]).copy()
df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())
df_clean = df_clean.dropna(subset=["embarked"]).reset_index(drop=True)

# Feature engineering (from NB10)
df_clean["family_size"] = df_clean["sibsp"] + df_clean["parch"] + 1
df_clean["is_alone"] = (df_clean["family_size"] == 1).astype(int)

# One-hot encoding (from NB09)
df_encoded = pd.get_dummies(
    df_clean,
    columns=["sex", "embarked"],
    drop_first=True,
    dtype=int,
)

features = ["age", "pclass", "sibsp", "parch", "fare",
            "family_size", "is_alone",
            "sex_male", "embarked_Q", "embarked_S"]
X = df_encoded[features]
y = df_encoded["survived"]

print(f"X: {X.shape}, y: {y.shape}")
print(f"Distribución del target: {y.value_counts().to_dict()}")

---

## 2. Regresión vs Clasificación — la diferencia conceptual

La **regresión lineal** del NB08 calculaba:

```
    fare ≈ β₀ + β₁·age + β₂·pclass + …
```

El problema: la salida es un número real. Para **survived = 0 ó 1** no nos sirve directamente — podríamos obtener cosas como `survived ≈ 1.34` o `survived ≈ -0.2`.

La **regresión logística** envuelve esa combinación lineal en una función **sigmoide**, que aplasta cualquier número real al intervalo **[0, 1]**:

```
    P(survived = 1) = sigmoid(β₀ + β₁·age + β₂·pclass + …)

         sigmoide:  σ(z) = 1 / (1 + e⁻ᶻ)
```

Ahora la salida es una **probabilidad**. La interpretas como: *"hay un 73% de probabilidad de que este pasajero haya sobrevivido"*.

Por defecto, se considera **clase 1** (sobrevivió) si la probabilidad ≥ 0.5, **clase 0** en caso contrario.

In [ ]:
# Visualize the sigmoid function — the heart of logistic regression
z = np.linspace(-8, 8, 200)
sigma = 1 / (1 + np.exp(-z))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(z, sigma, linewidth=2)
ax.axhline(0.5, color="red", linestyle="--", alpha=0.5, label="umbral 0.5")
ax.axvline(0, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("z  =  β₀ + β₁·x₁ + …  (combinación lineal)")
ax.set_ylabel("σ(z)  =  P(clase = 1)")
ax.set_title("Sigmoide — aplasta cualquier real a [0, 1]")
ax.legend()
plt.show()

---

## 3. Train/test split + escalado

Igual que en el NB11 — primero divides, luego escalas (¡fit sólo con train!).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,  # preserve class balance in both splits
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train_scaled: {X_train_scaled.shape}")
print(f"X_test_scaled:  {X_test_scaled.shape}")
print(f"\nProporción de sobrevivientes en train: {y_train.mean():.3f}")
print(f"Proporción de sobrevivientes en test:  {y_test.mean():.3f}")

> 💡 **`stratify=y`** asegura que ambos splits (train y test) conserven la **misma proporción** de cada clase. Sin esto, podrías tener mala suerte y obtener un test con muchos más sobrevivientes que el train — y evaluarías un escenario poco realista.

---

## 4. Entrenar la regresión logística

El API es **idéntico** al de la regresión lineal — fit y predict.

```python
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)
```

| Parámetro | Significado |
|---|---|
| `max_iter=1000` | Número máximo de iteraciones del optimizador. El default a veces no converge — subirlo es buena práctica. |

### 🏋️ Ejercicio 1 — Entrenar `LogisticRegression`

1. Importa `LogisticRegression` desde `sklearn.linear_model`.
2. Crea **`model`** = `LogisticRegression(max_iter=1000)`.
3. Entrénalo con `X_train_scaled` y `y_train`.

In [ ]:
from sklearn.linear_model import LogisticRegression

# YOUR CODE HERE
model = None

In [ ]:
# Tests — verify the exercise was completed correctly
assert isinstance(model, LogisticRegression), \
    f"model must be a LogisticRegression, got {type(model).__name__}"
assert hasattr(model, "coef_"), "model is not fitted — did you call .fit(X_train_scaled, y_train)?"

# coef_ shape: (1, n_features) for binary classification
assert model.coef_.shape == (1, 10), \
    f"Expected coef_ shape (1, 10), got {model.coef_.shape}"

# classes_ should be [0, 1]
assert set(model.classes_.tolist()) == {0, 1}, \
    f"Expected classes [0, 1], got {model.classes_.tolist()}"

print("✅ ¡Tests pasaron! El modelo está entrenado.")
print(f"\nIntercepto: {model.intercept_[0]:+.3f}")
print(f"\nCoeficientes aprendidos (después de escalado, son comparables entre sí):")
coef_series = pd.Series(model.coef_[0], index=X.columns).sort_values(key=abs, ascending=False)
for feature, c in coef_series.items():
    print(f"  {feature:15s} {c:+7.3f}")

### 🔍 Interpretar los coeficientes

Como **escalamos** las features antes, los coeficientes son **directamente comparables**: el de mayor valor absoluto es el más influyente.

- **Coeficiente positivo** → aumenta la probabilidad de sobrevivir.
- **Coeficiente negativo** → la disminuye.

Verás que **`sex_male`** suele tener el coeficiente más negativo (los hombres tenían mucha menos probabilidad de sobrevivir) y **`pclass`** también negativo (clases más altas — 3ª — peor probabilidad).

---

## 5. `.predict()` vs `.predict_proba()`

| Método | Devuelve | Cuándo usarlo |
|---|---|---|
| `.predict(X)` | Etiquetas predichas — array de 0/1 | Para obtener la **clase final** |
| `.predict_proba(X)` | Matriz `(n, 2)` con las probabilidades de cada clase | Para tener la **confianza** del modelo |

```python
predictions = model.predict(X_test_scaled)         # [0, 1, 1, 0, 0, …]
probas = model.predict_proba(X_test_scaled)        # [[0.8, 0.2], [0.3, 0.7], …]
                                                   #   ↑ P(class=0)  ↑ P(class=1)
```

> 💡 Cada fila de `predict_proba` **suma exactamente 1** (las dos probabilidades cubren todos los casos posibles).

### 🏋️ Ejercicio 2 — Calcular predicciones y probabilidades

1. Crea **`predictions`** = `model.predict(X_test_scaled)`.
2. Crea **`probas`** = `model.predict_proba(X_test_scaled)`.
3. Crea **`accuracy`** = proporción de predicciones correctas — `(predictions == y_test).mean()`.

In [ ]:
# YOUR CODE HERE
predictions = None
probas = None
accuracy = None

In [ ]:
# Tests — verify the exercise was completed correctly
assert predictions is not None and probas is not None and accuracy is not None, \
    "predictions, probas and accuracy must all be defined"

# predictions: 1D array of 0/1 with one entry per test row
assert predictions.shape == (178,), f"predictions shape must be (178,), got {predictions.shape}"
assert set(np.unique(predictions).tolist()) <= {0, 1}, "predictions must contain only 0/1"

# probas: (n, 2) matrix with rows summing to 1
assert probas.shape == (178, 2), f"probas shape must be (178, 2), got {probas.shape}"
row_sums = probas.sum(axis=1)
assert np.allclose(row_sums, 1, atol=1e-6), \
    "Each row of probas must sum to 1 (it's a probability distribution over the 2 classes)"

# Probabilities must be in [0, 1]
assert (probas >= 0).all() and (probas <= 1).all(), "probas values must be in [0, 1]"

# accuracy should be a float between 0 and 1
accuracy_float = float(accuracy)
assert 0 <= accuracy_float <= 1, f"accuracy must be in [0, 1], got {accuracy_float}"
# With this feature set, accuracy should be at least ~0.75
assert accuracy_float > 0.70, \
    f"Accuracy looks too low ({accuracy_float:.3f}) — expected > 0.70 with these features"

# Sanity: predictions correspond to argmax of probas (class with proba > 0.5)
expected_predictions = (probas[:, 1] >= 0.5).astype(int)
assert (predictions == expected_predictions).all(), \
    "predictions should be argmax of probas (class 1 if proba ≥ 0.5)"

print("✅ ¡Tests pasaron! Predicciones y probabilidades calculadas.")
print(f"\nAccuracy en test: {accuracy_float:.3f}  ({accuracy_float*100:.1f}% correctas)")
print(f"\nMuestra de las primeras 5 predicciones:")
print(pd.DataFrame({
    "y_real":     y_test.values[:5],
    "predicción": predictions[:5],
    "P(sobrevive)": probas[:5, 1].round(3),
}))

---

## 6. Visualizar las probabilidades

Una gran ventaja de `predict_proba` es que no todo es blanco o negro. Algunos pasajeros tienen **alta confianza** (proba cercana a 0 o a 1) y otros están **en la frontera** (alrededor de 0.5).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Color the histogram by true class
proba_died = probas[y_test.values == 0, 1]
proba_alive = probas[y_test.values == 1, 1]

ax.hist(proba_died, bins=20, alpha=0.6, label="Realmente murió", color="firebrick", edgecolor="black")
ax.hist(proba_alive, bins=20, alpha=0.6, label="Realmente sobrevivió", color="seagreen", edgecolor="black")
ax.axvline(0.5, color="black", linestyle="--", label="Umbral 0.5")
ax.set_xlabel("P(sobrevive) predicha por el modelo")
ax.set_ylabel("Frecuencia")
ax.set_title("Probabilidades predichas — separadas por la clase real")
ax.legend()
plt.show()

👀 ¿Qué deberías ver?

- La distribución roja (murieron) se concentra cerca de **0** — el modelo les asigna baja probabilidad de sobrevivir.
- La verde (sobrevivieron) se concentra cerca de **1**.
- Hay **superposición** en el medio — son los casos difíciles que el modelo confunde.

> 💡 **El umbral 0.5 es arbitrario**. Si te importa más detectar a los sobrevivientes (¡no quieres dejar a nadie!), podrías bajarlo a 0.3 — captarías más, a costa de más falsos positivos. En el NB13 verás cómo evaluar esto con precision y recall.

---

## 7. Resumen — ¿qué aprendiste?

🎉 ¡Acabas de entrenar tu primer **clasificador**! La estructura es **igualísima** a la regresión lineal — sólo cambias el modelo y aparece `predict_proba`.

### Conceptos clave

| Concepto | Idea |
|---|---|
| **Clasificación** | Predecir una **categoría** (vs un número en regresión) |
| **Sigmoide** | Aplasta cualquier real a [0, 1] → probabilidad |
| **`LogisticRegression`** | Modelo lineal + sigmoide |
| **`.predict_proba()`** | Devuelve la **confianza** (no solo la clase) |
| **`stratify=y`** | Mantiene la proporción de clases en train y test |

### Reglas prácticas

1. **Antes** de entrenar logística: encoding (NB09) + escalado (NB11). Si no escalas, los coeficientes no son comparables entre sí.
2. Usa **`stratify=y`** siempre que tu target sea categórico — especialmente si hay desequilibrio de clases.
3. **`max_iter=1000`** evita warnings de no-convergencia.
4. Usa **`.predict_proba`** cuando necesites la confianza (no solo la clase) — es información valiosa que `.predict` descarta.

### Lo que viene en el NB13

**Accuracy no es suficiente.** Si solo el 5% de un dataset tiene cáncer, predecir siempre "no cáncer" da 95% de accuracy pero es un modelo inútil. En el NB13 verás **precision, recall, F1, matriz de confusión y AUC** — la caja de herramientas completa para evaluar clasificadores.